# [IAPR][iapr]: Final project - Chocolate Recognition


**Moodle group ID:** 25  
**Kaggle challenge:**  `Deep learning` \
**Kaggle team name (exact):** "Image-inativi"  

**Author 1 (SCIPER):** Chiara Evangelisti (368672)   
**Author 2 (SCIPER):** Elisa Ferrara (371064)  
**Author 3 (SCIPER):** Francesco Maglie (378056)  

**Due date:** 21.05.2025 (11:59 pm)


## Key Submission Guidelines:
- **Before submitting your notebook, <span style="color:red;">rerun</span> it from scratch!** Go to: `Kernel` > `Restart & Run All`
- **Only groups of three will be accepted**, except in exceptional circumstances.


[iapr]: https://github.com/LTS5/iapr2025

---

1. [Introduction](#introduction)
2. [Data Augmentation](#data-augmentation)
    - 2.1 [Roboflow](#roboflow)  
    - 2.2 [Masked](#masked)  
3. [Pre-processing](#pre-processing)
    - 3.1 [Filters](#filters)  
    - 3.2 [Color modifications](#colors)
4. [Model](#model)
    - 4.1 [Initial model](#initial)  
    - 4.2 [Architecture modification](#architecture)
5. [Training](#training)
    - 5.1 [Training loop and optimizer](#loop)  
    - 5.2 [Losses](#losses)
    - 5.3 [Validation](#validation)
7. [Post-processing](#post-processing)
    - 6.1 [Connected components identification](#connected)  
    - 6.2 [Area- based separations](#area)
8. [Limitation and final considerations](#testing--inference)
9. [References](#references)


## 1) Introduction
<a id="introduction"></a>

Brief description of the problem, dataset, and goals.

## 2) Labeling and Data Augmentation
<a id="labeling-augmentation"></a>

Techniques used to increase dataset diversity (e.g., flipping, rotation, brightness, color jitter, Gaussian blur, etc.).

### 2.1 Roboflow
<a id="roboflow"></a>

The given training dataset contains only 90 images and the associated weak label, describing the number of instances per class in each image. Since the chosen network performs *semantic segmentation*, we need to modify the training dataset to manually add the mask and label of each instance. To do so, we used the online tool *Roboflow*, which, among other functionalities, let the user manually label the images in the training set.

We also decided to do **data augmentation** to increase the number of images in the training set. The applied transformations focus more on the position and orientation of the chocolates than their size and texture. For this reason, when exporting the dataset in Robotflow, we applied the following transformations:
- horizontal and vertical flip
- 90° rotation (clockwise, counter-clockwise, upside down)
- random rotation between -15° and +15°

Therefore, for each image in the training set, 3 additional images has been added, each one with one of the aforementioned transformation.

### 2.2 Masked
<a id="masked"></a>

## 3) Pre-processing
<a id="pre-processing"></a>

Transformations applied before feeding images to the model (e.g., resizing, normalization, mask handling).

### 3.1 Filters
<a id="filters"></a>

### 3.2 Color modifications
<a id="colors"></a>

## 4) Model
<a id="model"></a>

Model architecture definition (e.g., U-Net, ResNet-UNet, etc.), number of parameters, and discussion of key layers.

### 4.1 Initial model
<a id="initial"></a>

### 4.2 Architecture modifications
<a id="initialmodel"></a>

## 5) Training
<a id="training"></a>

Training loop, optimizer, loss functions (e.g., CrossEntropy, Dice), and learning rate strategy.


### 5.1 Training loop and optimizer
<a id="loop"></a>

### 5.2 Losses
<a id="losses"></a>

### 5.3) Validation
<a id="validation"></a>

Validation loop, metric tracking (e.g., IoU, accuracy, Dice score), and loss curves.

## 6) Chocolates count
<a id="chocolates-count"></a>

Since our network performs *semantic segmentation*, its output represents a mask in which a label from 0 (i.e. background) up to 13 has been assign to each pixel. The last remaining step consist of grouping each pixel belonging to the same instance, and classifying each chocolate.

### 6.1 Post processing
<a id="post-processing"></a>

Since the segmantation is not perfect, we need a simple post-processing pipeline to improve the predicted masks. This mainly aims to fill small holes and remove imperfections. The proposed pipeline consist of:
- image binarization: the mask is made binary to only distinguish foreground and background; it will be useful in the following steps
- `remove_small_holes()`: morphology-based scikit function to remove small holes
- `remove_small_objects()`: morphology-based scikit function to remove small imperfections

The morphology operator parameter has been set to have a mild effect and not to radically change the detected mask.

### 6.2 Count
<a id="count"></a>

Now that the segmented mask has been procesed, we need to focus on the count. Since each pixel has already been classified, we do not necessariy need an additional classifier to extract the whole class instance, but this can be done by a custom classical algorithm. The idea is to manually identify connected pixels that form an instance and use the most frequent label to select the chocolate class.
We preferred to perform the count task using a classical algorithm since it's straighforward given the masks. This let us use all the 12 million parameter to have a larger model solely dedicated to the semantic segmentation.

#### 6.2.1 Connected components
<a id="coneected-component"></a>

Since the network only classifies pixel and does not provide position information directly (e.g. bounding boxes), we can extract the connected component from the image to understand how many chocolates are present. To extract the connected component, we use the `label()` function provided by scikit: given a binary image, the function assigns the same numerical label to each pixel belonging to the same connected region.
Here there are a few examples of post-processing and subsequent connected component extraction.

<!-- ![binary_example_1](Images/1000777_2a.png)
![binary_example_3](Images/1010028_2a.png)
![binary_example_2](Images/1000978_2a.png) -->

<img src="Images/1000777_2a.png" alt="binary_example_1" width="70%">
<img src="Images/1010028_2a.png" alt="binary_example_2" width="70%">
<img src="Images/1000978_2a.png" alt="binary_example_3" width="70%">

As the third example image shows, there may be non-chocolate objects that the network falsely classify, but that are too large to be eliminated by the morphological operators. This problem is solved in the following stage of the pipeline (see below for more details).

Once we have a conencted component, we can use the pixel labels to decide which class should be assigned to the connected component: ideally, the most frequent label should be the most representative, and thus the class of the single chocolate.

Taking a look at the masks, we realized that this approach faces two major problems:
1. we are not able to correctly classify the instances if multiple classes are fused together in the same connected component, as only the larger chocolate (i.e. the one with more pixels) is always selected. In the example below, the second *Jelly Black* is discarded.
<!-- ![binary_example_2](Images/1000944_2a_p1.png) -->
<img src="Images/1000944_2a_p1.png" alt="prob1_2a" width="70%">

2. we are not able to classify multiple instances of the same class if they are fused in the same connected component, as the algorithm only counts one instance per connected region. In the following example, both the *Creme Brulee* and the *Jelly White* are counted only once.
<!-- ![binary_example_2](Images/1000789_2a_p2.png) -->
<img src="Images/1000789_2a_p2.png" alt="prob2_2a" width="70%">


#### 6.2.1 Area-based description
<a id="area-description"></a>

To solve the aforementioned problems, we decide to rely on a area-based descriptions: since, by design of the dataset, the images are always taken from a fixed distance from the table, the area of the chocolates should be consistent across all the images in the dataset. Therefore the idea is to compute the average area of each class representative (i.e. the average number of pixels) and use this information when comparing the number of pixel of each class detected in a connected component. This solves both problems.

It's worth mentioning that this approach also solves the problem related to the detection of non-chocolate objects that the network falsely classify: since the detected area is usually smaller that any chocolate (due to also the morphological-based post-processing), these connected components are discarded.

**STEP 1: Compute the area-based statistics**

To compute the average number of pixel per chocolate, we took the masks of the training set to calculate the number of pixel, and we extract the total amount of chocolates in the images using the weak labels. The relative script is `statistics.py` and the extracted information has been store in the main python file (which is used to generate the submission).

**STEP 2: The inference pipeline**

In this case, instead of considering the most frequent label, for each connected component:
- we count the number of pixel of each class
- we extract the number of chocolates of that class computing the ratio
$$
    n_i = \dfrac{p_i}{\bar{p}_i}
$$
where:
- $ n_i $ is the number of instance of class $i$ in the connected component
- $ p_i $ is the number of pixels classified as class $i$ in the connected component
- $ \bar{p}_i $ is the average number of pixels of an instance of class $i$

$n_i$ is then rounded to obtain an integer value (standard rules, `np.round()` has been used).

Despite its simplicity, this approach proves to be both reliable and effective, as it can been observed in the following example.

<img src="Images/1000944_2b_p1.png" alt="prob1_2a" width="70%">

**STEP 3: Scaling factor**

We can observe that, generally, the borders of the chocolates are those more affected by a wrong classification. While in most cases, this does not represent an issue, in certain images, the external misclassified border are quite wide and, as a consequence, the total area of the chocolate decreases, leading to an erroneous estimate of the total surface. This is visible in the following images, where the area of the connected component is too small and it is classified as a single instance instead of two.
<img src="Images/1000789_2a_p2.png" alt="prob2_2a" width="70%">

For this reason, a scaling factor $\alpha < 1$ is introduced and therefore the reference area of the class is slightly scaled down. Now the ratio to compute the number of chocolates of class $i$ becomes
$$
    n_i = \dfrac{p_i}{ \alpha \bar{p}_i}
$$
After some fine tuning, a value of $\alpha = 0.93$ has been found and yields appropriate results. As shown below, now the *Creme Brulee* and *Jelly White* instances are correctly counted.
<img src="Images/1000789_2b_p2_2.png" alt="prob2_2a" width="70%">


## 7) Limitation and final considerations
<a id="testing--inference"></a>

Running the trained model on test images, visualization of predictions, and export of results (e.g., CSV, overlay masks).


## 8) Reference
<a id="references"></a>

In [2]:
## YOUR CODE
...